# Module 01: Physics of Scalability Capacity Math — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/capacity_estimator.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import capacity_estimator

classes = [n for n, o in inspect.getmembers(capacity_estimator, inspect.isclass)
           if o.__module__ == 'capacity_estimator']
functions = [n for n, o in inspect.getmembers(capacity_estimator, inspect.isfunction)
             if o.__module__ == 'capacity_estimator']

print('module   : capacity_estimator')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(capacity_estimator, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Invalid parameters raise value error

This is the module's own `test_invalid_parameters_raise_value_error` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
import pytest
from capacity_estimator import CapacityEstimator

with pytest.raises(ValueError, match="DAU must be greater than zero"):
    CapacityEstimator(dau=0, read_ops_per_user_day=1, write_ops_per_user_day=1,
                      avg_read_payload_bytes=10, avg_write_payload_bytes=10)

with pytest.raises(ValueError, match="Operations per user cannot be negative"):
    CapacityEstimator(dau=100, read_ops_per_user_day=-1, write_ops_per_user_day=1,
                      avg_read_payload_bytes=10, avg_write_payload_bytes=10)

with pytest.raises(ValueError, match="Payload sizes must be positive integers"):
    CapacityEstimator(dau=100, read_ops_per_user_day=1, write_ops_per_user_day=1,
                      avg_read_payload_bytes=0, avg_write_payload_bytes=10)

print('PASSED: test_invalid_parameters_raise_value_error')

## 3. 🔮 Prediction — commit before you run

A single server handles 1,000 QPS at 40% CPU. Predict the QPS at which latency starts climbing sharply - and note it is *not* 2,500. What does queueing theory say happens as utilisation approaches 1?

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_qps_estimation_math`, which tests exactly this property.


In [ ]:
def standard_planner() -> CapacityEstimator:
    # 100M DAU, 10 reads, 1 write per user/day
    # Read size 2KB, Write size 500 bytes
    return CapacityEstimator(
        dau=100_000_000,
        read_ops_per_user_day=10,
        write_ops_per_user_day=1,
        avg_read_payload_bytes=2_000,
        avg_write_payload_bytes=500,
        peak_multiplier=2.0,
        retention_years=5,
        replication_factor=3,
        cache_coverage_ratio=0.20,
    )

_make_standard_planner = standard_planner

standard_planner = _make_standard_planner()

qps = standard_planner.estimate_qps()

# 100M * 10 reads / 86400 = 11,574.07
assert pytest.approx(qps.avg_read_qps, 0.1) == 11574.07
# 100M * 1 write / 86400 = 1,157.41
assert pytest.approx(qps.avg_write_qps, 0.1) == 1157.41
# Total QPS = 12,731.48
assert pytest.approx(qps.avg_total_qps, 0.1) == 12731.48
# Peak QPS = 2x
assert pytest.approx(qps.peak_total_qps, 0.1) == 25462.96

print('PASSED: test_qps_estimation_math')

## 4. Measure it: Bandwidth estimation math

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_bandwidth_estimation_math` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

def standard_planner() -> CapacityEstimator:
    # 100M DAU, 10 reads, 1 write per user/day
    # Read size 2KB, Write size 500 bytes
    return CapacityEstimator(
        dau=100_000_000,
        read_ops_per_user_day=10,
        write_ops_per_user_day=1,
        avg_read_payload_bytes=2_000,
        avg_write_payload_bytes=500,
        peak_multiplier=2.0,
        retention_years=5,
        replication_factor=3,
        cache_coverage_ratio=0.20,
    )

_make_standard_planner = standard_planner

standard_planner = _make_standard_planner()

bw = standard_planner.estimate_bandwidth()

# Peak write QPS = 2,314.82 * 500 bytes = ~1.16 MB/s -> ~9.26 Mbps
assert bw.ingress_mb_per_sec > 1.0
assert bw.ingress_mbps > 8.0

# Peak read QPS = 23,148.14 * 2,000 bytes = ~46.30 MB/s -> ~370.37 Mbps
assert bw.egress_mb_per_sec > 40.0
assert bw.egress_mbps > 320.0

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_bandwidth_estimation_math')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(capacity_estimator) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Latency does not degrade linearly - it degrades at a knee, near saturation.
2. Little's Law connects concurrency, throughput and latency; memorise it.
3. Vertical scaling has a ceiling you can compute in advance.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
